# 06 — LDA na amostra
Acompanho o amostra de 200 mil artigos, dos tokens até os vinte tópicos e o HTML.
Os parâmetros científicos são mostrados nas chamadas de treinamento.
`REPROCESSAR=False` abre os resultados atuais quando disponíveis; entradas ausentes são construídas.
Para mudar filtros ou parâmetros, use `REPROCESSAR=True` nos notebooks 04, 05 e 06.

In [1]:
from pathlib import Path
import sys
import os
import json
import time
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import FileLink, display

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root))
from src import config as cfg
from src.texto import preprocess, prepare_sample, token_records, save_corpus, lda_paths, save_lda_results
from gensim.corpora import Dictionary, MmCorpus
from gensim.models import LdaModel
from gensim.models.phrases import Phrases, FrozenPhrases
import pyLDAvis
import pyLDAvis.gensim_models

REPROCESSAR = False
AMOSTRA = True
cfg.ensure_directories()
pasta, modelos = lda_paths(AMOSTRA)
destino = cfg.OUTPUTS / ("amostras" if AMOSTRA else "completos") / "lda.html"

## 1. Sortear 200 mil IDs dos textos atuais
A população vem dos textos utilizáveis produzidos no notebook 04.
O sorteio é aleatório simples, sem reposição e com seed 42. Restauro a ordem original.
Não há reposição posterior dos documentos excluídos pelo vocabulário.

In [2]:
if REPROCESSAR or not (pasta / "05_tokens.parquet").is_file():
    prepare_sample(size=cfg.LDA_SAMPLE_SIZE, seed=cfg.RANDOM_SEED)
amostragem = json.loads((pasta / "amostragem.json").read_text(encoding="utf-8"))
display(pd.Series(amostragem).to_frame("Amostragem"))
fonte_tokens = pasta / "05_tokens.parquet"
ids = pd.read_parquet(pasta / "amostra_documentos.parquet")
display(ids.head())

,Amostragem
universo,506823
documentos,200000
seed,42
metodo,aleatoria_simples_sem_reposicao


,doc_index_origem,cord_uid
0,0,f9tg6xsg
1,2,1qgpa45q
2,6,44ip3jda
3,7,q259v3rr
4,11,tpaoi96c


## 2. Formar bigramas e filtrar o dicionário
Estes são os passos de construção do vocabulário. Expressões frequentes são unidas antes dos
cortes: pelo menos vinte documentos, no máximo 50% do conjunto e até 100 mil termos.
Na leitura de resultados atuais, carrego o dicionário e os bigramas já calculados.

In [3]:
construir = REPROCESSAR or not (pasta / "06_corpus.mm").is_file()
if construir:
    bigramas = FrozenPhrases(Phrases((palavras for _, palavras in token_records(fonte_tokens)),
                                    min_count=5, threshold=10))
    bigramas.save(str(pasta / "06_bigramas.phrases"))
    dicionario = Dictionary(palavras for _, palavras in token_records(fonte_tokens, bigramas))
    dicionario.filter_extremes(no_below=cfg.NO_BELOW, no_above=cfg.NO_ABOVE, keep_n=cfg.KEEP_N)
    dicionario.save(str(pasta / "06_dicionario.dict"))
else:
    dicionario = Dictionary.load(str(pasta / "06_dicionario.dict"))
print(f"Termos no dicionário: {len(dicionario):,}")

Termos no dicionário: 30,191


## 3. Converter os documentos em bag of words
Cada documento vira pares de identificador de termo e contagem. A gravação percorre os tokens
em blocos e registra os IDs na mesma ordem do corpus. Documentos sem vocabulário são separados.
O Matrix Market permanece no disco durante o treinamento.

In [4]:
if construir:
    corpus = save_corpus(dicionario, bigramas, sample=AMOSTRA)
else:
    corpus = MmCorpus(str(pasta / "06_corpus.mm"))
parametros = json.loads((pasta / "06_parametros.json").read_text(encoding="utf-8"))
display(pd.Series(parametros).to_frame("Corpus"))

,Corpus
no_below,20
no_above,0.5
keep_n,100000
vocabulario_antes,174724
vocabulario_depois,30191
documentos,199942
descartados_sem_vocabulario,58
leitura_em_lotes,True
sha256_tokens,811894dfc1ecfa9124f855892b14a5515de5f261a43028...


## 4. Treinar vinte tópicos
A chamada abaixo é o treinamento efetivo do LDA: dez passagens, seed 42 e minilotes de 2.000.
`update_every=1` atualiza o modelo após cada lote. Modelo, termos e métricas ficam na curadoria.
A log-perplexidade usa o próprio corpus de treinamento e não demonstra sozinha qualidade temática.

In [5]:
treinar = construir or not (modelos / "modelo_k20.model").is_file()
if treinar:
    inicio = time.perf_counter()
    modelo = LdaModel(corpus=corpus, id2word=dicionario, num_topics=cfg.LDA_NUM_TOPICS,
                      random_state=cfg.RANDOM_SEED, passes=cfg.LDA_PASSES,
                      chunksize=cfg.LDA_CHUNKSIZE, update_every=1)
    metricas = save_lda_results(modelo, dicionario, corpus, sample=AMOSTRA)
    print(f"Treinamento e métricas: {time.perf_counter() - inicio:.1f} segundos")
else:
    modelo = LdaModel.load(str(modelos / "modelo_k20.model"))
    metricas = json.loads((modelos / "metricas.json").read_text(encoding="utf-8"))
display(pd.DataFrame([metricas]).drop(columns="stopwords_adicionais", errors="ignore"))
display(pd.read_csv(modelos / "topicos_rotulos.csv"))

,num_topics,passes,random_state,documentos,vocabulario,log_perplexity
0,20,10,42,199942,30191,-9.094295


,topico,termos,rotulo_manual
0,0,"pandemic, country, time, impact, change, state...",NaN
1,1,"child, woman, adult, pregnancy, age, pregnant_...",NaN
2,2,"interaction, protein, target, structure, inhib...",NaN
3,3,"disease, increase, level, lung, effect, associ...",NaN
4,4,"test, positive, testing, sample, result, negat...",NaN
5,5,"pandemic, mental_health, work, stress, impact,...",NaN
6,6,"trial, group, intervention, evidence, result, ...",NaN
7,7,"high, risk, group, compare, mortality, datum, ...",NaN
8,8,"model, datum, base, method, analysis, result, ...",NaN
9,9,"high, mask, increase, food, low, reduce, devic...",NaN


## 5. Exportar somente o HTML do pyLDAvis
A visualização usa o modelo, o corpus e o dicionário acima. Mantenho a numeração dos tópicos
e a projeção MDS métrica. Nenhum PNG, SVG ou cópia duplicada é produzido.
O pyLDAvis usa bibliotecas externas de CDN ao abrir o HTML.

In [6]:
if treinar or not destino.is_file():
    visualizacao = pyLDAvis.gensim_models.prepare(modelo, corpus, dicionario,
        n_jobs=1, sort_topics=False, mds=cfg.PYLDAVIS_MDS,
        plot_opts={"xlab": "MDS 1", "ylab": "MDS 2"})
    pyLDAvis.save_html(visualizacao, str(destino))
display(FileLink(os.path.relpath(destino, Path.cwd())))

C:\UTFPR\TCC\tcc-data-viz\tcc-data-viz-main\outputs\amostras\lda.html